# Phase 2 — Trial Acquisition: Category Trial & Sub-brand Trial

**Purpose:** Quantify trial shopper acquisition by size for アリエールジェル vs アタック抗菌EX.

**Key Definitions:**
- **Category Trial** = Shopper's first-ever purchase in Laundry category (洗濯洗剤)
- **Sub-brand Trial** = Shopper's first-ever purchase of a specific sub-brand (e.g., ｱﾘｴｰﾙｼﾞｪﾙ)

| Step | Description |
|------|-------------|
| 2-1 | Count Category Trial & Sub-brand Trial per size per month |
| 2-2 | Compare trial volumes: アリエールジェル vs アタック抗菌EX (per size) |
| 2-3 | ASP elasticity: monthly ASP vs. sub-brand trial count per size |

**ASP = `SUM(pos_sales_amt) / SUM(pos_unit_sales_qty)`**  
**Created:** 2026-02-19

---
## 0. Imports & Connection

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
from dotenv import load_dotenv
import databricks.sql as sql
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

def _find_japanese_font():
    for name in ['MS Gothic', 'MS PGothic', 'Yu Gothic', 'Meiryo', 'IPAexGothic']:
        if name in {f.name for f in fm.fontManager.ttflist}:
            return name
    return None

_jp_font = _find_japanese_font()
if _jp_font:
    plt.rcParams['font.family'] = _jp_font
    print(f'✅ Japanese font: {_jp_font}')

load_dotenv(dotenv_path='../../.env')
DATABRICKS_HOST      = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN     = os.getenv('DATABRICKS_TOKEN')
DATABRICKS_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH')
assert all([DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_HTTP_PATH]), 'Missing .env credentials'
print('✅ Credentials loaded')

def execute_query(query: str) -> pd.DataFrame:
    with sql.connect(server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
                     access_token=DATABRICKS_TOKEN) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

✅ Japanese font: MS Gothic
✅ Credentials loaded


---
## 1. Parameters

In [2]:
ARIEL_GEL   = 'ｱﾘｴｰﾙｼﾞｪﾙ'
ATTACK_EX   = 'ｱﾀｯｸ抗菌EX'
SUB_CAT     = '洗濯洗剤'
CATEGORY    = 'Laundry'

# Full lookback for trial definition = earliest data available
LOOKBACK_START = '2024-01-01'  # Look back far enough to define "first purchase"
ANALYSIS_START = '2025-01-01'
ANALYSIS_END   = '2026-01-31'
RENEWAL_MONTH  = '2025-05-01'  # Update based on Phase 0

RETAILER_CODES = [
    'cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009',
    'cds_8010', 'cds_8011', 'cds_8012', 'cds_8013',
]
RETAILER_IN = ', '.join(f"'{c}'" for c in RETAILER_CODES)

print(f'📋 Lookback window for trial definition: {LOOKBACK_START}')
print(f'📋 Analysis window: {ANALYSIS_START} → {ANALYSIS_END}')
print(f'📋 Renewal breakpoint: {RENEWAL_MONTH}')

📋 Lookback window for trial definition: 2024-01-01
📋 Analysis window: 2025-01-01 → 2026-01-31
📋 Renewal breakpoint: 2025-05-01


---
## 2. Step 2-1: Category Trial & Sub-brand Trial Identification

**Logic:**
1. For each shopper, find their earliest purchase date in the **洗濯洗剤** sub-category (= category first instance)
2. For each shopper, find their earliest purchase date per **sub-brand** (= sub-brand first instance)
3. If the first instance falls within the analysis window (Jan 2025 - Jan 2026), they are a Trial shopper for that month

In [3]:
# ── Sub-brand Trial Shoppers per month per size ───────────────────────
# A shopper is a sub-brand trial if their FIRST-EVER purchase of that sub-brand
# falls in the analysis window, regardless of whether they bought other laundry brands before.

subbrand_trial_query = f"""
WITH shopper_first_purchase AS (
    -- Find each shopper's first purchase date per sub-brand + size
    SELECT
        idpos.shopper_key,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        prod.jp_segment_4_name            AS size_code,
        MIN(CAST(idpos.sales_period_group_end_date_part AS DATE)) AS first_purchase_date
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
           ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
           ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{LOOKBACK_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name IN ('{ARIEL_GEL}', '{ATTACK_EX}')
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3
),
-- Also find first-ever purchase at sub-brand level (any size)
shopper_first_subbrand AS (
    SELECT
        idpos.shopper_key,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        MIN(CAST(idpos.sales_period_group_end_date_part AS DATE)) AS first_subbrand_date
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
           ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
           ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{LOOKBACK_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name IN ('{ARIEL_GEL}', '{ATTACK_EX}')
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2
)
-- Count sub-brand trial shoppers whose first purchase is in the analysis window
SELECT
    DATE_TRUNC('month', sfp.first_purchase_date) AS trial_month,
    sfp.sub_brand,
    sfp.size_code,
    COUNT(DISTINCT CASE
        WHEN sfp.first_purchase_date = sfs.first_subbrand_date
        THEN sfp.shopper_key
    END) AS subbrand_trial_shoppers,
    COUNT(DISTINCT sfp.shopper_key) AS size_first_shoppers
FROM shopper_first_purchase sfp
LEFT JOIN shopper_first_subbrand sfs
       ON sfp.shopper_key = sfs.shopper_key
      AND sfp.sub_brand   = sfs.sub_brand
WHERE sfp.first_purchase_date BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""

print('⏳ Identifying sub-brand trial shoppers by size...', flush=True)
df_subbrand_trial = execute_query(subbrand_trial_query)
df_subbrand_trial['trial_month'] = pd.to_datetime(df_subbrand_trial['trial_month'])
for col in ['subbrand_trial_shoppers', 'size_first_shoppers']:
    df_subbrand_trial[col] = pd.to_numeric(df_subbrand_trial[col])

print(f'✅ {len(df_subbrand_trial)} rows fetched')
print(f'   Ariel trial rows: {len(df_subbrand_trial[df_subbrand_trial["sub_brand"]==ARIEL_GEL])}')
print(f'   Attack trial rows: {len(df_subbrand_trial[df_subbrand_trial["sub_brand"]==ATTACK_EX])}')

⏳ Identifying sub-brand trial shoppers by size...


HTTP request failed after retries: HTTPSConnectionPool(host='https', port=443): Max retries exceeded with url: //adb-2258763851730787.7.azuredatabricks.net/api/2.0/connector-service/feature-flags/PYTHON/4.2.3 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001D8ABC22710>: Failed to resolve 'https' ([Errno 11001] getaddrinfo failed)"))


✅ 182 rows fetched
   Ariel trial rows: 100
   Attack trial rows: 82


In [ ]:
# ── Category Trial Shoppers ───────────────────────────────────────────
# Shopper's first-ever purchase in 洗濯洗剤 sub-category

cat_trial_query = f"""
WITH shopper_first_category AS (
    SELECT
        idpos.shopper_key,
        MIN(CAST(idpos.sales_period_group_end_date_part AS DATE)) AS first_category_date
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
           ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
           ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{LOOKBACK_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1
),
-- Among category trial shoppers, what was their first-purchase sub-brand and size?
cat_trial_detail AS (
    SELECT
        idpos.shopper_key,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        prod.jp_segment_4_name            AS size_code,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
           ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
           ON idpos.shopper_key = shopper.shopper_key
    INNER JOIN shopper_first_category sfc
           ON idpos.shopper_key = sfc.shopper_key
          AND CAST(idpos.sales_period_group_end_date_part AS DATE) = sfc.first_category_date
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name IN ('{ARIEL_GEL}', '{ATTACK_EX}')
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
      AND sfc.first_category_date BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
)
SELECT
    DATE_TRUNC('month', ctd.purchase_date) AS trial_month,
    ctd.sub_brand,
    ctd.size_code,
    COUNT(DISTINCT ctd.shopper_key) AS category_trial_shoppers
FROM cat_trial_detail ctd
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""

print('⏳ Identifying category trial shoppers...', flush=True)
df_cat_trial = execute_query(cat_trial_query)
df_cat_trial['trial_month'] = pd.to_datetime(df_cat_trial['trial_month'])
df_cat_trial['category_trial_shoppers'] = pd.to_numeric(df_cat_trial['category_trial_shoppers'])

print(f'✅ {len(df_cat_trial)} rows')
print(f'   Ariel category trial: {df_cat_trial[df_cat_trial["sub_brand"]==ARIEL_GEL]["category_trial_shoppers"].sum():,.0f}')
print(f'   Attack category trial: {df_cat_trial[df_cat_trial["sub_brand"]==ATTACK_EX]["category_trial_shoppers"].sum():,.0f}')

⏳ Identifying category trial shoppers...


HTTP request error: 'NoneType' object has no attribute 'request'


✅ 179 rows
   Ariel category trial: 1,140,093
   Attack category trial: 2,156,911


HTTP request error: 'NoneType' object has no attribute 'request'
HTTP request error: 'NoneType' object has no attribute 'request'


In [5]:
# ── Merge both trial types ────────────────────────────────────────────
df_trial = df_subbrand_trial.merge(
    df_cat_trial,
    on=['trial_month', 'sub_brand', 'size_code'],
    how='outer'
).fillna(0)

for col in ['subbrand_trial_shoppers', 'size_first_shoppers', 'category_trial_shoppers']:
    df_trial[col] = df_trial[col].astype(int)

print('Combined Trial Summary (monthly):')
print('=' * 80)
summary = df_trial.groupby(['sub_brand', 'size_code']).agg(
    total_cat_trial=('category_trial_shoppers', 'sum'),
    total_subbrand_trial=('subbrand_trial_shoppers', 'sum'),
    total_size_first=('size_first_shoppers', 'sum'),
).reset_index()

print('\n▶ アリエールジェル')
print(summary[summary['sub_brand'] == ARIEL_GEL].to_string(index=False))
print('\n▶ アタック抗菌EX')
print(summary[summary['sub_brand'] == ATTACK_EX].to_string(index=False))

Combined Trial Summary (monthly):

▶ アリエールジェル
sub_brand     size_code  total_cat_trial  total_subbrand_trial  total_size_first
ｱﾘｴｰﾙｼﾞｪﾙ          本体通常           200610                540078            860703
ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大           329098                765118           1072319
ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           302753                618449           1083935
ｱﾘｴｰﾙｼﾞｪﾙ     詰替超ｼﾞｬﾝﾎﾞ             2982                  5122             12641
ｱﾘｴｰﾙｼﾞｪﾙ          詰替通常               51                   131               241
ｱﾘｴｰﾙｼﾞｪﾙ  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           183620                373038            677638
ｱﾘｴｰﾙｼﾞｪﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ           107559                217535            267389
ｱﾘｴｰﾙｼﾞｪﾙ           ｿﾉﾀ            13420                 28787             71425

▶ アタック抗菌EX
sub_brand     size_code  total_cat_trial  total_subbrand_trial  total_size_first
 ｱﾀｯｸ抗菌EX          本体通常           184008                464087            784099
 ｱﾀｯｸ抗菌EX         詰替超特大           505609           

---
## 3. Step 2-2: Trial Volume Comparison by Size

In [6]:
# ── Sub-brand trial: monthly trend per size, Ariel vs Attack ──────────
for brand_name, brand_code in [('アリエールジェル', ARIEL_GEL), ('アタック抗菌EX', ATTACK_EX)]:
    brand_data = df_trial[df_trial['sub_brand'] == brand_code].copy()
    sizes = sorted(brand_data['size_code'].unique())

    if len(sizes) == 0:
        print(f'⚠️ No data for {brand_name}')
        continue

    n_s = len(sizes)
    n_c = min(2, n_s)
    n_r = (n_s + n_c - 1) // n_c

    fig = make_subplots(rows=n_r, cols=n_c,
                        subplot_titles=[f'{brand_name} {s}' for s in sizes],
                        vertical_spacing=0.1, horizontal_spacing=0.08)

    for idx, size in enumerate(sizes):
        row = idx // n_c + 1
        col = idx % n_c + 1
        subset = brand_data[brand_data['size_code'] == size].sort_values('trial_month')

        fig.add_trace(go.Bar(x=subset['trial_month'], y=subset['subbrand_trial_shoppers'],
                             name=f'{size} Sub-brand Trial', marker_color='#FF6B6B',
                             showlegend=(idx==0)),
                      row=row, col=col)
        fig.add_trace(go.Bar(x=subset['trial_month'], y=subset['category_trial_shoppers'],
                             name=f'{size} Category Trial', marker_color='#4ECDC4',
                             showlegend=(idx==0)),
                      row=row, col=col)

    fig.update_layout(height=300*n_r, barmode='group',
                      title_text=f'{brand_name}: Monthly Trial Shoppers by Size (Category vs Sub-brand)',
                      template='plotly_white')
    fig.update_yaxes(title_text='Trial Shoppers')
    fig.show()

In [7]:
# ── Head-to-head: Ariel vs Attack sub-brand trial by size ─────────────
# Total sub-brand trial shoppers per size over the full period
h2h = df_trial.groupby(['sub_brand', 'size_code'])['subbrand_trial_shoppers'].sum().reset_index()

fig_h2h = px.bar(h2h, x='size_code', y='subbrand_trial_shoppers', color='sub_brand',
                 barmode='group', color_discrete_map={ARIEL_GEL: '#1E90FF', ATTACK_EX: '#FF6347'},
                 labels={'subbrand_trial_shoppers': 'Sub-brand Trial Shoppers', 'size_code': 'Size'},
                 title='Head-to-Head: Sub-brand Trial Shoppers by Size (Jan 2025 - Jan 2026)')
fig_h2h.update_layout(template='plotly_white', height=500)
fig_h2h.show()

---
## 2b. Monthly Trial Comparison: アリエールジェル vs アタック抗菌EX per Size

In [12]:
# ── Monthly trial comparison: Ariel vs Attack per size (side-by-side) ─
# Get common sizes between both brands
ariel_sizes_set = set(df_trial[df_trial['sub_brand'] == ARIEL_GEL]['size_code'].unique())
attack_sizes_set = set(df_trial[df_trial['sub_brand'] == ATTACK_EX]['size_code'].unique())
common_sizes = sorted(ariel_sizes_set & attack_sizes_set)

# Filter to major sizes (exclude ｿﾉﾀ, 詰替通常, 詰替超ｼﾞｬﾝﾎﾞ if tiny)
major_sizes = [s for s in common_sizes if
               df_trial[(df_trial['size_code'] == s)]['subbrand_trial_shoppers'].sum() > 500]

n_s = len(major_sizes)
n_c = min(2, n_s)
n_r = (n_s + n_c - 1) // n_c

fig = make_subplots(rows=n_r, cols=n_c,
                    subplot_titles=[f'{s}' for s in major_sizes],
                    vertical_spacing=0.12, horizontal_spacing=0.08)

for idx, size in enumerate(major_sizes):
    row = idx // n_c + 1
    col = idx % n_c + 1

    for brand_code, brand_label, color in [
        (ARIEL_GEL,  'Ariel', '#1E90FF'),
        (ATTACK_EX,  'Attack', '#FF6347')
    ]:
        subset = df_trial[(df_trial['sub_brand'] == brand_code) &
                          (df_trial['size_code'] == size)].sort_values('trial_month')
        fig.add_trace(go.Scatter(
            x=subset['trial_month'], y=subset['subbrand_trial_shoppers'],
            mode='lines+markers', name=brand_label,
            line=dict(color=color), marker=dict(size=6),
            showlegend=(idx == 0)
        ), row=row, col=col)

    fig.update_yaxes(title_text='Trial Shoppers', row=row, col=col)

fig.update_layout(
    height=350 * n_r,
    title_text='Monthly Sub-brand Trial: アリエールジェル vs アタック抗菌EX (per Size)',
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=-0.15)
)
fig.show()

# ── Print monthly summary table ──────────────────────────────────────
print('\nMonthly Trial Comparison (major sizes):')
for size in major_sizes:
    ariel_total = df_trial[(df_trial['sub_brand'] == ARIEL_GEL) &
                           (df_trial['size_code'] == size)]['subbrand_trial_shoppers'].sum()
    attack_total = df_trial[(df_trial['sub_brand'] == ATTACK_EX) &
                            (df_trial['size_code'] == size)]['subbrand_trial_shoppers'].sum()
    ratio = attack_total / ariel_total if ariel_total > 0 else 0
    print(f'  {size:30s} Ariel: {ariel_total:>8,}  Attack: {attack_total:>8,}  Ratio: {ratio:.2f}x')


Monthly Trial Comparison (major sizes):
  本体通常                           Ariel:  540,078  Attack:  464,087  Ratio: 0.86x
  詰替超特大                          Ariel:  765,118  Attack:  869,455  Ratio: 1.14x
  詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                  Ariel:  618,449  Attack: 1,356,215  Ratio: 2.19x
  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                   Ariel:  373,038  Attack:  652,050  Ratio: 1.75x
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ                    Ariel:  217,535  Attack:  427,545  Ratio: 1.97x
  ｿﾉﾀ                            Ariel:   28,787  Attack:    7,309  Ratio: 0.25x


---
## 4. Step 2-3: ASP Elasticity — Does Lower ASP Drive More Trials?

In [13]:
# ── Fetch WEEKLY RETAILER-LEVEL ASP + units per size ──────────────────
# Weekly × retailer granularity is the standard approach for price elasticity
# — monthly national averages smooth out too much variation
asp_weekly_query = f"""
SELECT
    CAST(idpos.sales_period_group_end_date_part AS DATE) AS week_end,
    idpos.data_provider_code_part AS retailer,
    prod.jp_sub_brand_alter_lang_name   AS sub_brand,
    prod.jp_segment_4_name              AS size_code,
    SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS weighted_asp,
    SUM(idpos.pos_unit_sales_qty) AS total_units,
    COUNT(DISTINCT idpos.shopper_key) AS buyer_count
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
       ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
       ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN})
  AND prod.jp_category_name = '{CATEGORY}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{ARIEL_GEL}', '{ATTACK_EX}')
  AND shopper.member_ind = 'Y'
  AND idpos.pos_unit_sales_qty > 0
GROUP BY 1, 2, 3, 4
ORDER BY 1, 2, 3, 4
"""

print('⏳ Fetching weekly retailer-level ASP + units (more granular than monthly)...', flush=True)
df_asp_weekly = execute_query(asp_weekly_query)
df_asp_weekly['week_end'] = pd.to_datetime(df_asp_weekly['week_end'])
for c in ['weighted_asp', 'total_units', 'buyer_count']:
    df_asp_weekly[c] = pd.to_numeric(df_asp_weekly[c])
print(f'✅ {len(df_asp_weekly):,} rows (weekly × retailer × brand × size)')
print(f'   Weeks: {df_asp_weekly["week_end"].nunique()}, Retailers: {df_asp_weekly["retailer"].nunique()}')

# Also keep monthly ASP for backward compat (aggregate from weekly)
df_asp = df_asp_weekly.copy()
df_asp['month'] = df_asp['week_end'].dt.to_period('M').dt.to_timestamp()
df_asp = df_asp.groupby(['month', 'sub_brand', 'size_code']).agg(
    weighted_asp=('weighted_asp', 'mean'),
    total_units=('total_units', 'sum')
).reset_index()

⏳ Fetching weekly retailer-level ASP + units (more granular than monthly)...
✅ 29,713 rows (weekly × retailer × brand × size)
   Weeks: 396, Retailers: 9


In [15]:
# ── Merge weekly ASP with monthly trial counts ────────────────────────
# Trial is defined monthly; aggregate weekly ASP to monthly for this scatter
df_asp_monthly_agg = df_asp_weekly.groupby(
    [df_asp_weekly['week_end'].dt.to_period('M').dt.to_timestamp().rename('trial_month'),
     'sub_brand', 'size_code']
).agg(
    weighted_asp=('weighted_asp', 'mean'),
    total_units=('total_units', 'sum')
).reset_index()

# Ensure tz-naive for merge compatibility
df_trial_compat = df_trial.copy()
if hasattr(df_trial_compat['trial_month'].dt, 'tz') and df_trial_compat['trial_month'].dt.tz is not None:
    df_trial_compat['trial_month'] = df_trial_compat['trial_month'].dt.tz_localize(None)

df_elasticity = df_trial_compat.merge(
    df_asp_monthly_agg,
    on=['trial_month', 'sub_brand', 'size_code'],
    how='inner'
)

# ── Plot: Weekly retailer-level ASP vs weekly buyer count (Ariel) ─────
# This shows the scatter at weekly × retailer granularity
ariel_weekly = df_asp_weekly[df_asp_weekly['sub_brand'] == ARIEL_GEL].copy()
ariel_w_sizes = sorted(ariel_weekly['size_code'].unique())
# Filter to sizes with enough data
ariel_w_sizes = [s for s in ariel_w_sizes if len(ariel_weekly[ariel_weekly['size_code'] == s]) >= 10]

n_s = len(ariel_w_sizes)
n_c = min(2, n_s)
n_r = (n_s + n_c - 1) // n_c

fig = make_subplots(rows=n_r, cols=n_c,
                    subplot_titles=[f'{s}: Weekly Retailer ASP vs Buyers' for s in ariel_w_sizes],
                    vertical_spacing=0.12, horizontal_spacing=0.1)

for idx, size in enumerate(ariel_w_sizes):
    row = idx // n_c + 1
    col = idx % n_c + 1
    subset = ariel_weekly[ariel_weekly['size_code'] == size]

    if len(subset) >= 10:
        corr = subset['weighted_asp'].corr(subset['buyer_count'])
        fig.add_trace(go.Scatter(
            x=subset['weighted_asp'], y=subset['buyer_count'],
            mode='markers', name=f'{size} (r={corr:.2f})',
            marker=dict(size=5, opacity=0.5),
            showlegend=True),
            row=row, col=col)

        # Add trendline
        z = np.polyfit(subset['weighted_asp'], subset['buyer_count'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(subset['weighted_asp'].min(), subset['weighted_asp'].max(), 50)
        fig.add_trace(go.Scatter(x=x_line, y=p(x_line), mode='lines',
                                 line=dict(dash='dash', color='red'), showlegend=False),
                      row=row, col=col)

    fig.update_xaxes(title_text='ASP (JPY)', row=row, col=col)
    fig.update_yaxes(title_text='Buyers (weekly×retailer)', row=row, col=col)

fig.update_layout(height=350*n_r,
                  title_text='アリエールジェル: Weekly×Retailer ASP Elasticity (More Granular)',
                  template='plotly_white')
fig.show()

# ── Print correlation table ───────────────────────────────────────────
print('\nASP vs Buyer Count Correlation by Size (weekly × retailer, Ariel):')
for size in ariel_w_sizes:
    s = ariel_weekly[ariel_weekly['size_code'] == size]
    if len(s) >= 10:
        corr = s['weighted_asp'].corr(s['buyer_count'])
        n = len(s)
        print(f'  {size:30s} r = {corr:+.3f}  (n={n:,} obs) {"price drop → more buyers" if corr < 0 else "price not key driver"}')


ASP vs Buyer Count Correlation by Size (weekly × retailer, Ariel):
  本体通常                           r = -0.445  (n=3,448 obs) price drop → more buyers
  詰替超特大                          r = -0.379  (n=3,424 obs) price drop → more buyers
  詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                  r = -0.024  (n=2,867 obs) price drop → more buyers
  詰替超ｼﾞｬﾝﾎﾞ                      r = +0.496  (n=1,458 obs) price not key driver
  詰替通常                           r = +0.440  (n=108 obs) price not key driver
  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                   r = -0.243  (n=3,469 obs) price drop → more buyers
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ                    r = -0.434  (n=396 obs) price drop → more buyers
  ｿﾉﾀ                            r = -0.497  (n=815 obs) price drop → more buyers


## 5. Price Gap Heatmaps (50 JPY bins)
Each heatmap pairs **Ariel** and **Attack 抗菌EX** ASPs by week × retailer, floored to 50 JPY bands.
- **Heatmap A** — Value = Ariel buyer count (proxy for trial acquisition velocity)
- **Heatmap B** — Value = Ariel unit index vs Attack (Ariel units ÷ Attack units × 100)

In [16]:
# ── Price Gap Heatmaps per Size ────────────────────────────────────────
import plotly.express as px

# Pair Ariel and Attack at weekly × retailer × size level
ariel_w = df_asp_weekly[df_asp_weekly['sub_brand'] == ARIEL_GEL][
    ['week_end', 'retailer', 'size_code', 'weighted_asp', 'total_units', 'buyer_count']
].copy()
attack_w = df_asp_weekly[df_asp_weekly['sub_brand'] == ATTACK_EX][
    ['week_end', 'retailer', 'size_code', 'weighted_asp', 'total_units']
].copy()

paired = ariel_w.merge(
    attack_w, on=['week_end', 'retailer', 'size_code'],
    suffixes=('_ariel', '_attack'), how='inner'
)
print(f'Paired observations (week × retailer × size): {len(paired):,}')

# Floor ASPs to 50 JPY bins
paired['ariel_asp_bin'] = (paired['weighted_asp_ariel'] // 50 * 50).astype(int)
paired['attack_asp_bin'] = (paired['weighted_asp_attack'] // 50 * 50).astype(int)

# Unit index: Ariel units / Attack units × 100
paired['unit_index'] = paired['total_units_ariel'] / paired['total_units_attack'] * 100

sizes_for_hm = sorted(paired['size_code'].unique())

# ═══ Heatmap A: Ariel Buyer Count per price combination ══════════════
print('\n' + '='*80)
print('Heatmap A: Price Gap × Ariel Buyer Count (weekly × retailer obs.)')
print('='*80)

for size in sizes_for_hm:
    s = paired[paired['size_code'] == size]
    if len(s) < 5:
        print(f'  Skipping {size}: too few observations ({len(s)})')
        continue

    # Pivot: rows=Attack ASP bin, cols=Ariel ASP bin, value=sum of Ariel buyers
    hm = s.pivot_table(
        index='attack_asp_bin', columns='ariel_asp_bin',
        values='buyer_count', aggfunc='sum', fill_value=0
    ).sort_index(ascending=False)

    fig = px.imshow(
        hm, text_auto=True, aspect='auto',
        labels=dict(x='アリエール ASP (50JPY bin)', y='アタック抗菌EX ASP (50JPY bin)',
                    color='Ariel Buyers'),
        title=f'【{size}】Price Gap × Ariel Buyer Count',
        color_continuous_scale='YlOrRd'
    )
    fig.update_layout(width=750, height=550)
    fig.show()

# ═══ Heatmap B: Ariel Unit Index vs Attack per price combination ═════
print('\n' + '='*80)
print('Heatmap B: Price Gap × Ariel Unit Index vs Attack')
print('='*80)

for size in sizes_for_hm:
    s = paired[paired['size_code'] == size]
    if len(s) < 5:
        print(f'  Skipping {size}: too few observations ({len(s)})')
        continue

    # Pivot: rows=Attack ASP bin, cols=Ariel ASP bin, value=mean unit index
    hm = s.pivot_table(
        index='attack_asp_bin', columns='ariel_asp_bin',
        values='unit_index', aggfunc='mean', fill_value=np.nan
    ).sort_index(ascending=False)

    fig = px.imshow(
        hm, text_auto='.0f', aspect='auto',
        labels=dict(x='アリエール ASP (50JPY bin)', y='アタック抗菌EX ASP (50JPY bin)',
                    color='Unit Index (Ariel/Attack×100)'),
        title=f'【{size}】Price Gap × Ariel Unit Index vs Attack',
        color_continuous_scale='RdYlGn'
    )
    fig.update_layout(width=750, height=550)
    fig.show()

# ── Summary table: optimal price combinations ────────────────────────
print('\n' + '='*80)
print('Optimal Price Combinations (highest Ariel buyer count per size)')
print('='*80)
for size in sizes_for_hm:
    s = paired[paired['size_code'] == size]
    if len(s) < 5:
        continue
    top = s.groupby(['ariel_asp_bin', 'attack_asp_bin']).agg(
        total_buyers=('buyer_count', 'sum'),
        mean_unit_index=('unit_index', 'mean'),
        obs_count=('week_end', 'count')
    ).reset_index().sort_values('total_buyers', ascending=False).head(5)

    print(f'\n▶ {size}:')
    print(f'  {"Ariel ASP":>12s} {"Attack ASP":>12s} {"Buyers":>10s} {"Unit Idx":>10s} {"Obs":>6s}')
    for _, r in top.iterrows():
        print(f'  {int(r["ariel_asp_bin"]):>10,}~ {int(r["attack_asp_bin"]):>10,}~ '
              f'{int(r["total_buyers"]):>10,} {r["mean_unit_index"]:>10.0f} {int(r["obs_count"]):>6,}')

Paired observations (week × retailer × size): 13,429

Heatmap A: Price Gap × Ariel Buyer Count (weekly × retailer obs.)



Heatmap B: Price Gap × Ariel Unit Index vs Attack



Optimal Price Combinations (highest Ariel buyer count per size)

▶ 本体通常:
     Ariel ASP   Attack ASP     Buyers   Unit Idx    Obs
         200~        350~    360,625       3772    108
         150~        200~    170,752        575     73
         150~        300~    164,417        933     70
         200~        300~    101,173        298    128
         250~        200~    100,840        105    118

▶ 詰替超特大:
     Ariel ASP   Attack ASP     Buyers   Unit Idx    Obs
         300~        350~    982,245       1526    582
         300~        300~    626,652        248    467
         350~        350~    616,599        118    600
         250~        350~    479,227        229    123
         300~        400~    340,976       2204    225

▶ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ:
     Ariel ASP   Attack ASP     Buyers   Unit Idx    Obs
         850~        800~    550,316         66    225
         850~        750~    427,394         46    214
         850~        700~    372,982         40    214
         750

In [17]:
# ── Export Phase 2 results ────────────────────────────────────────────
output_file = 'phase2_trial_acquisition.xlsx'

# Strip timezone for Excel compatibility
def strip_tz(df):
    df = df.copy()
    for col in df.select_dtypes(include=['datetimetz']).columns:
        df[col] = df[col].dt.tz_localize(None)
    # Also convert Period / Interval types to string
    for col in df.columns:
        if df[col].dtype == 'object':
            try:
                df[col] = df[col].astype(str)
            except:
                pass
    return df

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    strip_tz(df_trial).to_excel(writer, sheet_name='Trial_Monthly', index=False)
    strip_tz(summary).to_excel(writer, sheet_name='Trial_Summary', index=False)
    strip_tz(df_elasticity).to_excel(writer, sheet_name='ASP_Elasticity', index=False)
    strip_tz(df_asp_weekly).to_excel(writer, sheet_name='Weekly_ASP_Retailer', index=False)
    strip_tz(paired).to_excel(writer, sheet_name='Price_Gap_Paired', index=False)

print(f'✅ Phase 2 results exported to {output_file}')
print(f'   Sheets: Trial_Monthly, Trial_Summary, ASP_Elasticity, Weekly_ASP_Retailer, Price_Gap_Paired')

✅ Phase 2 results exported to phase2_trial_acquisition.xlsx
   Sheets: Trial_Monthly, Trial_Summary, ASP_Elasticity, Weekly_ASP_Retailer, Price_Gap_Paired
